In [15]:
#import python packages
import os
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import glob
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy.optimize import least_squares, differential_evolution
from scipy.integrate import solve_ivp
import time as timer

#import data loading and pre-processing functions from separate .py files
from data_loading_v5 import define_metadata, import_exp_data, filter_dataframes, process_dataframes, convert_reader_data, subtract,normalize 
from plot_timecourses import plottimecourselist, plottimecoursearray, figure_layout 
from crosstalk import crosstalk, mixed_crosstalk, antibiotic_crosstalk
from dose_response_fitting import dose_response_fitting
from updated_mechanistic_model import difeq_newest_test_updated, fp_total_timecourse, sensor_fit, plot_fittings
from VAE import VAE, train, test, warmup_scheduler, get_latent_variables, count_parameters
from VAEMLP import MLP, CombinedModel, combo_train, validate

RUN_TAG = "newmech_diff_params"
tag = f"_{RUN_TAG}" if RUN_TAG else ""

# ============================================================
# IMPROVED FITTING FUNCTIONS (inline instead of separate file)
# ============================================================

def sensor_fit_improved(sensors, samples, tspan, inputs, alpha, K, hill, 
                       algorithm='trf', bounds_version='restrictive'):
    """
    Improved sensor fitting with configurable bounds and algorithms.
    """
    t0 = timer.perf_counter()
    
    inits_train = samples[:, 0::len(tspan)]
    samples_flat = samples.ravel()
    n = sensors
    
    # Reference parameters
    mu_mean = 0.756
    ds_mean = 0.2
    r0_mean = 0.3
    k_mean = 1
    ks_mean = 0.198/2
    theta_mean = 3
    
    # CRITICAL: Different dp0 bounds based on version
    if bounds_version == 'restrictive':
        dp_mean = 0.05
        bounds_lower = [0.1, 0, 0, 0.0001, 0.02, 0.01, 0.6] * n
        bounds_upper = [2, 0.5, 1, 1, 0.2, 1, 6] * n
        print("✓ Using RESTRICTIVE bounds: dp0=[0.02, 0.2]")
        
    elif bounds_version == 'moderate':
        dp_mean = 0.08
        bounds_lower = [0.1, 0, 0, 0.0001, 0.03, 0.01, 0.6] * n
        bounds_upper = [2, 0.8, 1, 1, 0.3, 1, 6] * n
        print("✓ Using MODERATE bounds: dp0=[0.03, 0.3]")
        
    else:  # flexible
        dp_mean = 0.1
        bounds_lower = [0.1, 0, 0, 0.0001, 0, 0.01, 0.6] * n
        bounds_upper = [2, 1, 1, 1, 1, 1, 6] * n
        print("✓ Using FLEXIBLE bounds: dp0=[0, 1]")
    
    para_ref = np.array([mu_mean, ds_mean, r0_mean, k_mean, dp_mean, 
                        ks_mean, theta_mean] * n)
    bounds = np.array([bounds_lower, bounds_upper])
    
    def residual_function(params, y_data, time_range, p_0, inputs, alpha, K, hill):
        params_reshape = params.reshape(n, 7)
        results = []
        
        for i in range(len(p_0)):
            od_0 = p_0[i, 0] / n
            fluor_0 = p_0[i, 1:] / od_0
            yinit = np.zeros(2*n)
            yinit[:n] = od_0
            yinit[n:] = fluor_0
            s = inputs[i]
            
            sol = solve_ivp(
                lambda t, y: difeq_newest_test_updated(t, y, params_reshape, 
                                                       alpha, K, hill, s, n),
                [time_range[0], time_range[-1]], yinit, t_eval=time_range
            )
            
            results.append(fp_total_timecourse(sol.y, n, n).ravel())
        
        results = np.hstack(results)
        residuals = results - y_data
        return residuals
    
    if algorithm == 'differential_evolution':
        print("✓ Using DIFFERENTIAL EVOLUTION (global optimizer)")
        
        def objective(params):
            residuals = residual_function(params, samples_flat, tspan, inits_train, 
                                         inputs, alpha, K, hill)
            return np.sum(residuals**2)
        
        result = differential_evolution(
            objective,
            bounds=list(zip(bounds[0], bounds[1])),
            maxiter=100,
            popsize=15,
            seed=42,
            disp=True
        )
        sens_popt = result.x
        cost = result.fun
        nfev = result.nfev
        
    else:
        print(f"✓ Using {algorithm.upper()} algorithm")
        result = least_squares(
            fun=residual_function,
            x0=para_ref,
            bounds=bounds,
            method=algorithm,
            args=(samples_flat, tspan, inits_train, inputs, alpha, K, hill),
            verbose=2,
            max_nfev=300
        )
        sens_popt = result.x
        cost = result.cost
        nfev = result.nfev
    
    t1 = timer.perf_counter()
    print(f"✓ Fitting time = {t1 - t0:.0f} s")
    
    return sens_popt, cost, nfev


def verify_fitted_parameters(all_params_test, sensors, mu_mean=0.756):
    """Comprehensive parameter verification."""
    print("\n" + "="*70)
    print("FITTED PARAMETER VERIFICATION:")
    print("="*70)
    
    params_reshaped = all_params_test.reshape(sensors, 7)
    
    fitted_mu = params_reshaped[:, 0]
    fitted_ds = params_reshaped[:, 1]
    fitted_r0 = params_reshaped[:, 2]
    fitted_k = params_reshaped[:, 3]
    fitted_dp0 = params_reshaped[:, 4]
    fitted_Ks = params_reshaped[:, 5]
    fitted_theta = params_reshaped[:, 6]
    
    print(f"\nNumber of sensors: {sensors}")
    print("\n--- Growth Parameters ---")
    print(f"μ (growth rate):     {fitted_mu}")
    print(f"  Mean: {np.mean(fitted_mu):.4f}, Range: [{np.min(fitted_mu):.4f}, {np.max(fitted_mu):.4f}]")
    
    print(f"\nds (input burden):   {fitted_ds}")
    print(f"  Mean: {np.mean(fitted_ds):.4f}, Range: [{np.min(fitted_ds):.4f}, {np.max(fitted_ds):.4f}]")
    
    print("\n--- Protein Decay Parameters (KEY CHECK) ---")
    print(f"dp0 (non-dilution decay): {fitted_dp0}")
    print(f"  Mean: {np.mean(fitted_dp0):.4f}, Range: [{np.min(fitted_dp0):.4f}, {np.max(fitted_dp0):.4f}]")
    
    half_life_stationary = np.log(2) / (fitted_dp0 + 1e-10)
    half_life_exponential = np.log(2) / (np.mean(fitted_mu) + fitted_dp0)
    
    print(f"\nProtein half-lives (stationary phase):")
    print(f"  Mean: {np.mean(half_life_stationary):.2f} hours")
    print(f"\nProtein half-lives (exponential phase):")
    print(f"  Mean: {np.mean(half_life_exponential):.2f} hours")
    
    # Sanity checks
    print("\n--- Sanity Checks ---")
    warnings = []
    
    if np.any(fitted_dp0 < 0.01):
        warnings.append("⚠️  Some dp0 < 0.01 (near lower bound)")
    if np.any(fitted_dp0 > 0.25):
        warnings.append("⚠️  Some dp0 > 0.25 (near upper bound)")
    if np.any(half_life_stationary > 100):
        warnings.append("⚠️  Some proteins have half-life > 100 hours")
    
    if warnings:
        print("\n".join(warnings))
    else:
        print("✅ All parameters look reasonable!")
    
    print("\n" + "="*70)
    
    return params_reshaped


# ============================================================
# MAIN PIPELINE STARTS HERE
# ============================================================

#select which microbial community dataset to work with
community='aTc_IPTG'

#if using antibiotic data, indicate the plasmid and inhibitor combination
plasmid='HSGBla'
inhibitor='TAZ'
if community!='antibiotic_data':
    plasmid=None
    inhibitor=None

#import metadata for selected community
files,readers, fluors, fluor1, fluor2, fluor3, single_file, sensor_names, sensors,time_vector,od_raws,input_arrays,input_names,fp1_raws,fp2_raws,fp3_raws = define_metadata(community,plasmid,inhibitor)

if community=='antibiotic_data':
    od_conv=od_raws
    fp1_conv=fp1_raws
    fp2_conv=fp2_raws
    fp3_conv=None
else:
    od_conv=convert_reader_data(readers, None, od_raws,community)
    fp1_conv=convert_reader_data(readers, fluor1, fp1_raws,community)
    fp2_conv=convert_reader_data(readers, fluor2, fp2_raws,community)
    if sensors==3:
        fp3_conv=convert_reader_data(readers, fluor3, fp3_raws,community)
    else:
        fp3_conv=None

# Subtract basal expression
subtracted_fp1_conv_all, subtracted_fp2_conv_all, subtracted_fp3_conv_all=subtract(community,time_vector,fp1_conv,fp2_conv,fp3_conv,sensors,input_arrays)

# Min-max scale
normalized_fp1_conv_all, normalized_fp2_conv_all, normalized_fp3_conv_all=normalize(subtracted_fp1_conv_all,subtracted_fp2_conv_all,subtracted_fp3_conv_all)

#Append the full OD and fluorescence timecourses
od_stack=np.empty((0,len(time_vector)), dtype=float)
for i, reader in enumerate(fluor1):
    od_stack=np.vstack((od_stack,od_conv[i]))

concat_list = [od_stack]
concat_list.append(normalized_fp1_conv_all)
concat_list.append(normalized_fp2_conv_all)
if normalized_fp3_conv_all is not None and normalized_fp3_conv_all.size>0:
    concat_list.append(normalized_fp3_conv_all)
exp_data_new=np.concatenate(concat_list,axis=1)
    
exp_inputs=np.vstack(list(input_arrays.values()))

# Load pre-calculated parameters
timepoint=20 
if community=='antibiotic_data':
    save_dir = f"parameter_files/{community}/{plasmid}_{inhibitor}/"
else:
    save_dir = f"parameter_files/{community}/"

alpha_files=glob.glob(f'{save_dir}*_{timepoint}hr_{community}_alphas_mixed.npy')
alphas=np.load(max(alpha_files))

hill_timepoint=20 
K_files=glob.glob(f'{save_dir}*_{hill_timepoint}hr_{community}_K_calc.npy')
K_calc=np.load(max(K_files))
    
hill_files=glob.glob(f'{save_dir}*_{hill_timepoint}hr_{community}_hill_coef.npy')
hill_calc=np.load(max(hill_files))

# Calculate limits
true_K=np.zeros(sensors)
upper=np.zeros(sensors)
lower=np.zeros(sensors)
for i in range(sensors):
    true_K[i]=K_calc[i*sensors + i] 
    upper[i]=true_K[i]*(99**(1/hill_calc[i*sensors + i]))
    lower[i]=true_K[i]/(99**(1/hill_calc[i*sensors + i])) 

# Filter data
if ((community=='cuma_ohc_atc')|(community=='van_dapg_nar')|(community=='antibiotic_data')):
    mask=(exp_inputs>0).all(axis=1)
else:
    mask = (exp_inputs >= lower).all(axis=1) & (exp_inputs <= upper).all(axis=1) 

filtered_exp_inputs = exp_inputs[mask]
filtered_exp_data_new = exp_data_new[mask]

# Normalize
normalized_exp_inputs = filtered_exp_inputs / true_K
normalized_K_calc=K_calc.reshape((sensors,sensors))/true_K


# ============================================================
# PARAMETER FITTING WITH MULTIPLE ALGORITHMS
# ============================================================

print("\n" + "="*70)
print("ALTERNATIVE-FIT ANALYSIS")
print("Generating multiple parameter sets using different algorithms")
print(f"Results will be saved with tag: {tag}")
print("="*70)

alpha_reshaped = alphas.reshape((sensors, sensors))
hill_reshaped = hill_calc.reshape((sensors, sensors))

# Store all parameter sets
all_parameter_sets = {}
all_costs = {}

# Configuration
fit_configurations = [
    ('trf', 'restrictive', 'TRF_Restrictive'),
    ('dogbox', 'moderate', 'Dogbox_Moderate'),
    ('trf', 'moderate', 'TRF_Moderate'),
]

# Run fitting
for algorithm, bounds_version, name in fit_configurations:
    print(f"\n{'='*70}")
    print(f"Fitting with: {name}")
    print(f"Algorithm: {algorithm}, Bounds: {bounds_version}")
    print(f"{'='*70}\n")
    
    params, cost, nfev = sensor_fit_improved(
        sensors, 
        filtered_exp_data_new, 
        time_vector,
        normalized_exp_inputs, 
        alpha_reshaped, 
        normalized_K_calc, 
        hill_reshaped,
        algorithm=algorithm,
        bounds_version=bounds_version
    )
    
    all_parameter_sets[name] = params
    all_costs[name] = cost
    
    print(f"\n>>> Verifying {name} parameters:")
    params_reshaped = verify_fitted_parameters(params, sensors)
    
    # Save with RUN_TAG
    if community == 'antibiotic_data':
        save_dir = f"parameter_files/{community}/{plasmid}_{inhibitor}/"
    else:
        save_dir = f"parameter_files/{community}/"
    
    os.makedirs(save_dir, exist_ok=True)
    datestamp = datetime.now().strftime("%Y-%m-%d")
    
    np.save(f"{save_dir}{datestamp}{tag}_params_{name}.npy", params)
    np.save(f"{save_dir}{datestamp}{tag}_cost_{name}.npy", cost)
    print(f"✓ Saved: {save_dir}{datestamp}{tag}_params_{name}.npy")

# Print comparison
print("\n" + "="*70)
print("FIT QUALITY COMPARISON:")
print("="*70)
for name, cost in all_costs.items():
    print(f"  {name:30s}: Cost = {cost:.6f}")
print("="*70 + "\n")

print(f"✅ Successfully generated {len(all_parameter_sets)} parameter sets!")
print(f"   All saved with tag: {tag}")


# ============================================================
# GENERATE SIMULATIONS FOR EACH PARAMETER SET
# ============================================================

from scipy.stats import truncnorm

print("\n" + "="*70)
print("GENERATING 10K SIMULATIONS FOR EACH PARAMETER SET")
print("="*70)

# Calculate initial conditions
inits_train = filtered_exp_data_new[:, 0::len(time_vector)]

od_0_mean = np.mean(inits_train[:, 0]) / sensors
od_0_std = np.std(inits_train[:, 0]) / sensors
od_0_max = np.max(inits_train[:, 0]) / sensors
od_0_min = np.min(inits_train[:, 0]) / sensors

fluor_stats = []
for j in range(sensors):
    od_0 = inits_train[:, 0] / sensors
    fluor_mean = np.mean(inits_train[:, j+1] / od_0)
    fluor_std = np.std(inits_train[:, j+1] / od_0)
    fluor_min = np.min(inits_train[:, j+1] / od_0)
    fluor_max = np.max(inits_train[:, j+1] / od_0)
    fluor_stats.append((fluor_mean, fluor_std, fluor_min, fluor_max))

def get_truncated_normal(mean=0, sd=1, low=0, upp=10):
    return truncnorm((low - mean) / sd, (upp - mean) / sd, loc=mean, scale=sd)

num_simulations = 10000

# Generate same inputs for all parameter sets
np.random.seed(42)
s_full = (10 ** ((np.random.rand(num_simulations, sensors) * 
          (np.log10(upper) - np.log10(lower))) + np.log10(lower))) / true_K

# Generate simulations
all_simulations = {}
all_y0_values = {}

for param_name, all_params_test in all_parameter_sets.items():
    print(f"\n>>> Generating simulations for: {param_name}")
    
    optimized_params = all_params_test.reshape(sensors, 7)
    time_courses = np.empty((num_simulations, sensors+1, len(time_vector)))
    y0_values = []
    
    for i in range(num_simulations):
        if (i+1) % 2000 == 0:
            print(f"    Progress: {i+1}/{num_simulations}")
        
        s = s_full[i]
        
        OD0 = get_truncated_normal(mean=od_0_mean, sd=od_0_std, 
                                   low=od_0_min, upp=od_0_max).rvs()
        
        fluor_values = [
            get_truncated_normal(mean=stats[0], sd=stats[1], 
                               low=stats[2], upp=stats[3]).rvs()
            for stats in fluor_stats
        ]
        
        y0 = np.hstack(([OD0] * sensors, fluor_values))
        
        sol = solve_ivp(
            difeq_newest_test_updated,
            [time_vector[0], time_vector[-1]],
            y0,
            t_eval=time_vector,
            args=(optimized_params, alpha_reshaped, normalized_K_calc, 
                  hill_reshaped, s, sensors)
        )
        
        time_courses[i] = fp_total_timecourse(sol.y, sensors, sensors)
        y0_values.append(y0)
    
    all_simulations[param_name] = time_courses
    all_y0_values[param_name] = np.array(y0_values)
    
    # Save with RUN_TAG
    if community == 'antibiotic_data':
        save_dir = f"parameter_files/{community}/{plasmid}_{inhibitor}/"
    else:
        save_dir = f"parameter_files/{community}/"
    
    os.makedirs(save_dir, exist_ok=True)
    
    np.savez(f"{save_dir}{datestamp}{tag}_{community}_curves10k_{param_name}.npz", 
             time_courses)
    np.save(f"{save_dir}{datestamp}{tag}_{community}_S_10k_{param_name}.npy", s_full)
    np.save(f"{save_dir}{datestamp}{tag}_{community}_y0_10k_{param_name}.npy", 
            np.array(y0_values))
    
    print(f"✓ Saved: {save_dir}{datestamp}{tag}_{community}_curves10k_{param_name}.npz")

print("\n" + "="*70)
print("✅ ALL SIMULATIONS GENERATED SUCCESSFULLY")
print(f"   Total simulations: {len(all_parameter_sets)} x 10,000 = {len(all_parameter_sets) * 10000}")
print(f"   All files saved with tag: {tag}")
print("="*70 + "\n")

print("\nREADY FOR VAE-MLP TRAINING!")
print(f"Next: Train VAE-MLP on each parameter set's simulations")
print(f"Files to use:")
for param_name in all_parameter_sets.keys():
    print(f"  - {datestamp}{tag}_{community}_curves10k_{param_name}.npz")


ALTERNATIVE-FIT ANALYSIS
Generating multiple parameter sets using different algorithms
Results will be saved with tag: _newmech_diff_params

Fitting with: TRF_Restrictive
Algorithm: trf, Bounds: restrictive

✓ Using RESTRICTIVE bounds: dp0=[0.02, 0.2]
✓ Using TRF algorithm
   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         1.3725e+03                                    7.57e+03    
       1              2         7.1606e+02      6.56e+02       6.38e-01       4.42e+03    
       2              3         4.7137e+02      2.45e+02       4.03e-01       1.80e+03    
       3              4         4.2152e+02      4.98e+01       4.90e-01       4.18e+03    
       4              5         3.5917e+02      6.23e+01       1.15e-01       9.28e+01    
       5              6         3.3666e+02      2.25e+01       5.09e-01       1.74e+01    
       6              7         3.2974e+02      6.92e+00       7.34e-01       3.83e+02  

In [16]:
"""
COMPLETE PIPELINE - Part 2: VAE-MLP Training and Comparison

Add this after the simulation generation code.
This section trains VAE-MLP for each parameter set and compares predictions.
"""

# ============================================================
# VAE-MLP TRAINING FOR EACH PARAMETER SET
# ============================================================

print("\n" + "="*70)
print("TRAINING SEPARATE VAE-MLP FOR EACH PARAMETER SET")
print("This demonstrates parameter non-uniqueness doesn't affect predictions")
print("="*70)

import time as timer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

# Hyperparameters
batch_size = 32
latent_dim = 10
latent_channel = 6
alpha_vae = 1e-4
lr = 1e-3
min_lr = 5e-6
epochs = 400
gamma = 0.99
weight_decay = 1e-5
warmup_epochs = 8
patience = 30
hidden_size = 128

# Store all trained models
trained_models = {}
training_histories = {}

# Train VAE-MLP for each parameter set
for param_name in all_parameter_sets.keys():
    print(f"\n{'='*70}")
    print(f"TRAINING VAE-MLP FOR: {param_name}")
    print(f"{'='*70}\n")
    
    t0 = timer.perf_counter()
    
    # Load simulations for this parameter set
    if community == 'antibiotic_data':
        save_dir = f"parameter_files/{community}/{plasmid}_{inhibitor}/"
    else:
        save_dir = f"parameter_files/{community}/"
    
    data_array = np.load(f"{save_dir}{datestamp}{tag}_{community}_curves10k_{param_name}.npz")['arr_0']
    s_full_loaded = np.load(f"{save_dir}{datestamp}{tag}_{community}_S_10k_{param_name}.npy")
    
    # Flatten data
    data_concat = data_array.reshape(data_array.shape[0], -1)
    seq_length = data_concat.shape[1]
    
    # Combine with experimental data
    combined_data_np = np.vstack((data_concat, filtered_exp_data_new))
    
    # Create indices to locate experimental data
    exp_data_indices = range(data_concat.shape[0], 
                             data_concat.shape[0] + filtered_exp_data_new.shape[0])
    
    # Normalize
    scaler = MinMaxScaler()
    data_normalized_np = scaler.fit_transform(combined_data_np)
    data_normalized = torch.tensor(data_normalized_np).float().unsqueeze(1)
    
    # Prepare inputs
    combined_inputs = np.vstack((s_full_loaded, normalized_exp_inputs))
    log_inputs = np.log10(combined_inputs)
    log_inputs_tensor = torch.from_numpy(log_inputs).float()
    
    # Split train/test (SAME split for all models for fair comparison)
    train_data, test_data, train_labels, test_labels, train_idx, test_idx = train_test_split(
        data_normalized, log_inputs_tensor, 
        range(data_normalized.shape[0]), 
        test_size=0.2, random_state=42
    )
    
    # ============================================================
    # TRAIN VAE
    # ============================================================
    
    print(f">>> Training VAE...")
    
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
    
    torch.manual_seed(42)
    vae_model = VAE(latent_dim=latent_dim, latent_channel=latent_channel, seq_length=seq_length)
    vae_model = vae_model.to(device)
    
    print(f'VAE has {count_parameters(vae_model):,} parameters')
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(vae_model.parameters(), lr=lr, weight_decay=weight_decay)
    
    train_loss_values = []
    test_loss_values = []
    
    best_test_loss = np.inf
    epochs_no_improve = 0
    
    scheduler1 = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda epoch: warmup_scheduler(epoch, warmup_epochs))
    scheduler2 = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    
    for epoch in range(epochs):
        train_loss = train(vae_model, train_loader, optimizer, criterion, 
                          alpha_vae, device, latent_channel, seq_length)
        test_loss = test(vae_model, test_loader, criterion, device, 
                        latent_channel, seq_length)
        train_loss_values.append(train_loss)
        test_loss_values.append(test_loss)
        
        # Clamp minimum learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = max(param_group['lr'], min_lr)
        
        interval = 2 if epoch < 10 else 40
        if (epoch + 1) % interval == 0:
            print(f'Epoch: {epoch + 1} Train: {train_loss:.7f}, Test: {test_loss:.7f}, Lr:{param_group["lr"]:.8f}')
        
        # Update learning rate
        if epoch < warmup_epochs:
            scheduler1.step()
        else:
            scheduler2.step()
        
        # Early stopping
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve == patience:
            print('Early stopping!')
            break
    
    # Save VAE model
    if community == 'antibiotic_data':
        model_save_dir = f"final_trained_models/{community}/{plasmid}_{inhibitor}/"
    else:
        model_save_dir = f"final_trained_models/{community}/"
    
    os.makedirs(model_save_dir, exist_ok=True)
    vae_path = f"{model_save_dir}{datestamp}{tag}_VAE_{community}_{param_name}.pt"
    torch.save(vae_model.state_dict(), vae_path)
    print(f"✓ Saved VAE: {vae_path}")
    
    # ============================================================
    # TRAIN MLP
    # ============================================================
    
    print(f"\n>>> Training MLP...")
    
    # Create MLP
    mlp_model = MLP(latent_dim, hidden_size, log_inputs_tensor.shape[1]).to(device)
    
    # Load the trained VAE
    vae_model_for_combo = VAE(latent_dim, latent_channel, seq_length).to(device)
    vae_model_for_combo.load_state_dict(torch.load(vae_path))
    
    # Create combined model
    combined_model = CombinedModel(vae_model_for_combo, mlp_model).to(device)
    
    print(f'MLP has {count_parameters(mlp_model):,} parameters')
    print(f'Combined model has {count_parameters(combined_model):,} parameters')
    
    # Prepare data loaders
    train_dataset = TensorDataset(train_data, train_labels)
    train_loader_mlp = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataset = TensorDataset(test_data, test_labels)
    test_loader_mlp = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(mlp_model.parameters(), lr=lr, weight_decay=weight_decay)
    
    best_test_loss = np.inf
    epochs_no_improve = 0
    
    scheduler1 = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda epoch: warmup_scheduler(epoch, warmup_epochs))
    scheduler2 = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)
    
    train_loss_values_mlp = []
    test_loss_values_mlp = []
    
    for epoch in range(epochs):
        train_loss = combo_train(combined_model, train_loader_mlp, optimizer, 
                                criterion, device, latent_channel, seq_length)
        test_loss = validate(combined_model, test_loader_mlp, criterion, 
                           device, latent_channel, seq_length)
        train_loss_values_mlp.append(train_loss)
        test_loss_values_mlp.append(test_loss)
        
        # Clamp minimum learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = max(param_group['lr'], min_lr)
        
        interval = 2 if epoch < 10 else 40
        if (epoch + 1) % interval == 0:
            print(f'Epoch: {epoch + 1} Train: {train_loss:.7f}, Test: {test_loss:.7f}, Lr:{param_group["lr"]:.8f}')
        
        # Update learning rate
        if epoch < warmup_epochs:
            scheduler1.step()
        else:
            scheduler2.step()
        
        # Early stopping
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve == patience:
            print('Early stopping!')
            break
    
    # Save combined model
    combined_path = f"{model_save_dir}{datestamp}{tag}_VAEMLP_{community}_{param_name}.pt"
    torch.save(combined_model.state_dict(), combined_path)
    print(f"✓ Saved VAE-MLP: {combined_path}")
    
    # Store for later comparison
    trained_models[param_name] = {
        'model': combined_model,
        'scaler': scaler,
        'train_idx': train_idx,
        'test_idx': test_idx,
        'exp_data_indices': exp_data_indices,
        'vae_train_loss': train_loss_values,
        'vae_test_loss': test_loss_values,
        'mlp_train_loss': train_loss_values_mlp,
        'mlp_test_loss': test_loss_values_mlp
    }
    
    training_histories[param_name] = {
        'vae_train': train_loss_values,
        'vae_test': test_loss_values,
        'mlp_train': train_loss_values_mlp,
        'mlp_test': test_loss_values_mlp
    }
    
    t1 = timer.perf_counter()
    print(f"\n✓ Total training time for {param_name}: {(t1-t0)/60:.1f} minutes")

print("\n" + "="*70)
print("✅ ALL VAE-MLPs TRAINED SUCCESSFULLY")
print(f"   Trained {len(trained_models)} models")
print("="*70 + "\n")


# ============================================================
# COMPARE PREDICTIONS ON EXPERIMENTAL DATA
# ============================================================

print("\n" + "="*70)
print("COMPARING PREDICTIONS ON EXPERIMENTAL DATA")
print("This demonstrates parameter non-uniqueness doesn't affect predictions")
print("="*70)

# Store predictions from each model
all_predictions = {}
all_r2_scores = {}

for param_name, model_info in trained_models.items():
    print(f"\n>>> Testing {param_name} on experimental data...")
    
    model = model_info['model']
    scaler = model_info['scaler']
    
    # Prepare experimental data
    exp_data_scaled = scaler.transform(filtered_exp_data_new)
    exp_data_tensor = torch.tensor(exp_data_scaled).float().unsqueeze(1).to(device)
    
    # Get predictions
    with torch.no_grad():
        predictions = model(exp_data_tensor, latent_channel, seq_length)
    
    predictions_np = predictions.cpu().numpy()
    
    # Convert from log space to original concentrations
    predicted_concentrations = true_K * 10**predictions_np
    true_concentrations = normalized_exp_inputs * true_K
    
    all_predictions[param_name] = predicted_concentrations
    
    # Calculate R² for each input
    r2_scores = []
    for i in range(sensors):
        r2 = r2_score(np.log10(true_concentrations[:, i]), 
                     np.log10(predicted_concentrations[:, i]))
        r2_scores.append(r2)
    
    all_r2_scores[param_name] = r2_scores
    
    print(f"  R² scores:")
    for i, (name, r2) in enumerate(zip(input_names, r2_scores)):
        print(f"    {name}: R² = {r2:.4f}")


# ============================================================
# VISUALIZATION 1: R² Score Comparison
# ============================================================

print("\n>>> Creating comparison figures...")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

x_pos = np.arange(sensors)
width = 0.8 / len(trained_models)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, (param_name, r2_scores) in enumerate(all_r2_scores.items()):
    offset = (idx - len(trained_models)/2) * width + width/2
    ax.bar(x_pos + offset, r2_scores, width, label=param_name, 
           alpha=0.8, color=colors[idx % len(colors)])

ax.set_xlabel('Input', fontsize=12, fontweight='bold')
ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
ax.set_title('Prediction Performance Across Different Parameter Sets', 
            fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(input_names)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1])

# Add horizontal line at R²=0.9
ax.axhline(y=0.9, color='red', linestyle='--', linewidth=2, alpha=0.5, label='R²=0.9')

plt.tight_layout()

if community == 'antibiotic_data':
    save_dir = f"figures/{community}/{plasmid}_{inhibitor}/"
else:
    save_dir = f"figures/{community}/"

os.makedirs(save_dir, exist_ok=True)
fig.savefig(f"{save_dir}r2_comparison_across_paramsets{tag}.svg", format='svg')
plt.close()

print(f"✓ Saved: {save_dir}r2_comparison_across_paramsets{tag}.svg")


# ============================================================
# VISUALIZATION 2: Prediction Scatter Plots
# ============================================================

n_models = len(trained_models)
fig, axes = plt.subplots(sensors, n_models, 
                        figsize=(4*n_models, 4*sensors))

if sensors == 1:
    axes = axes.reshape(1, -1)

for model_idx, (param_name, predictions) in enumerate(all_predictions.items()):
    for sensor_idx in range(sensors):
        ax = axes[sensor_idx, model_idx]
        
        true_vals = np.log10(normalized_exp_inputs[:, sensor_idx] * true_K[sensor_idx])
        pred_vals = np.log10(predictions[:, sensor_idx])
        
        # Scatter plot
        ax.scatter(true_vals, pred_vals, alpha=0.6, s=30, color='blue')
        
        # y=x line
        min_val = min(true_vals.min(), pred_vals.min())
        max_val = max(true_vals.max(), pred_vals.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
        
        # Labels and title
        if sensor_idx == 0:
            ax.set_title(param_name, fontsize=12, fontweight='bold')
        if model_idx == 0:
            ax.set_ylabel(f'Predicted log({input_names[sensor_idx]})', fontsize=10)
        if sensor_idx == sensors - 1:
            ax.set_xlabel(f'True log({input_names[sensor_idx]})', fontsize=10)
        
        # R² annotation
        r2 = all_r2_scores[param_name][sensor_idx]
        ax.text(0.05, 0.95, f'R² = {r2:.3f}', 
               transform=ax.transAxes, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(f"{save_dir}prediction_scatter_comparison{tag}.svg", format='svg')
plt.close()

print(f"✓ Saved: {save_dir}prediction_scatter_comparison{tag}.svg")


# ============================================================
# VISUALIZATION 3: Prediction Agreement Heatmap
# ============================================================

model_names = list(all_predictions.keys())
n_models = len(model_names)

fig, axes = plt.subplots(1, sensors, figsize=(5*sensors, 5))
if sensors == 1:
    axes = [axes]

for sensor_idx in range(sensors):
    ax = axes[sensor_idx]
    
    # Create correlation matrix
    corr_matrix = np.ones((n_models, n_models))
    
    for i, name1 in enumerate(model_names):
        for j, name2 in enumerate(model_names):
            if i != j:
                pred1 = np.log10(all_predictions[name1][:, sensor_idx])
                pred2 = np.log10(all_predictions[name2][:, sensor_idx])
                corr_matrix[i, j] = np.corrcoef(pred1, pred2)[0, 1]
    
    # Plot heatmap
    im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=0.9, vmax=1.0)
    
    # Add text annotations
    for i in range(n_models):
        for j in range(n_models):
            text = ax.text(j, i, f'{corr_matrix[i, j]:.3f}',
                         ha="center", va="center", color="black", fontsize=10)
    
    ax.set_xticks(range(n_models))
    ax.set_yticks(range(n_models))
    ax.set_xticklabels(model_names, rotation=45, ha='right')
    ax.set_yticklabels(model_names)
    ax.set_title(f'{input_names[sensor_idx]}: Prediction Correlation', 
                fontsize=12, fontweight='bold')
    
    plt.colorbar(im, ax=ax, label='Correlation')

plt.tight_layout()
fig.savefig(f"{save_dir}prediction_correlation_heatmap{tag}.svg", format='svg')
plt.close()

print(f"✓ Saved: {save_dir}prediction_correlation_heatmap{tag}.svg")


# ============================================================
# VISUALIZATION 4: Training Loss Curves
# ============================================================

fig, axes = plt.subplots(2, len(trained_models), figsize=(5*len(trained_models), 8))

if len(trained_models) == 1:
    axes = axes.reshape(-1, 1)

for idx, (param_name, history) in enumerate(training_histories.items()):
    # VAE losses
    ax = axes[0, idx]
    ax.semilogy(history['vae_train'], label='Train', linewidth=2)
    ax.semilogy(history['vae_test'], label='Test', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'VAE Loss - {param_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # MLP losses
    ax = axes[1, idx]
    ax.semilogy(history['mlp_train'], label='Train', linewidth=2)
    ax.semilogy(history['mlp_test'], label='Test', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'MLP Loss - {param_name}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(f"{save_dir}training_loss_comparison{tag}.svg", format='svg')
plt.close()

print(f"✓ Saved: {save_dir}training_loss_comparison{tag}.svg")


# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("\n" + "="*70)
print("SUMMARY: Prediction Agreement Analysis")
print("="*70)

for sensor_idx in range(sensors):
    print(f"\n{input_names[sensor_idx]}:")
    print(f"  R² scores across models:")
    
    r2_values = [all_r2_scores[name][sensor_idx] for name in model_names]
    print(f"    Mean: {np.mean(r2_values):.4f}")
    print(f"    Std:  {np.std(r2_values):.4f}")
    print(f"    Range: [{np.min(r2_values):.4f}, {np.max(r2_values):.4f}]")
    
    # Calculate average pairwise correlation
    correlations = []
    for i, name1 in enumerate(model_names):
        for j, name2 in enumerate(model_names):
            if i < j:
                pred1 = np.log10(all_predictions[name1][:, sensor_idx])
                pred2 = np.log10(all_predictions[name2][:, sensor_idx])
                corr = np.corrcoef(pred1, pred2)[0, 1]
                correlations.append(corr)
    
    print(f"  Average pairwise prediction correlation: {np.mean(correlations):.4f}")

print(f"\n" + "="*70)
print("CONCLUSION:")
print("="*70)

avg_std = np.mean([np.std([all_r2_scores[name][i] for name in model_names]) 
                   for i in range(sensors)])

if avg_std < 0.05:
    print("✅ Low variability in R² scores across parameter sets!")
    print("✅ Different parameter sets yield similar prediction performance!")
    print("✅ Parameter non-uniqueness does NOT affect input predictions!")
    print("\nThis demonstrates that despite having different ODE parameters,")
    print("the VAE-MLP models all make similar predictions on experimental data.")
    print("Therefore, parameter non-uniqueness is NOT a problem for this application!")
else:
    print("⚠️  Significant variability in predictions across parameter sets.")
    print("   Consider investigating parameter sensitivity further.")

print("="*70 + "\n")

print("\n🎉 ALTERNATIVE-FIT ANALYSIS COMPLETE!")
print(f"   Generated {len(all_parameter_sets)} parameter sets")
print(f"   Trained {len(trained_models)} VAE-MLP models")
print(f"   All results saved with tag: {tag}")
print("\n" + "="*70)


TRAINING SEPARATE VAE-MLP FOR EACH PARAMETER SET
This demonstrates parameter non-uniqueness doesn't affect predictions
Using device: cuda


TRAINING VAE-MLP FOR: TRF_Restrictive

>>> Training VAE...
VAE has 159,597 parameters
Epoch: 2 Train: 0.0049708, Test: 0.0022965, Lr:0.00025000
Epoch: 4 Train: 0.0014469, Test: 0.0007422, Lr:0.00050000
Epoch: 6 Train: 0.0014240, Test: 0.0004002, Lr:0.00075000
Epoch: 8 Train: 0.0007531, Test: 0.0007667, Lr:0.00100000
Epoch: 10 Train: 0.0006375, Test: 0.0004155, Lr:0.00099000
Epoch: 40 Train: 0.0003297, Test: 0.0001314, Lr:0.00073230
Epoch: 80 Train: 0.0002552, Test: 0.0001495, Lr:0.00048989
Epoch: 120 Train: 0.0002389, Test: 0.0000730, Lr:0.00032772
Epoch: 160 Train: 0.0002216, Test: 0.0000740, Lr:0.00021924
Early stopping!
✓ Saved VAE: final_trained_models/aTc_IPTG/2026-01-13_newmech_diff_params_VAE_aTc_IPTG_TRF_Restrictive.pt

>>> Training MLP...
MLP has 18,178 parameters
Combined model has 177,775 parameters


/tmp/ipykernel_1883563/687396089.py:168: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vae_model_for_combo.load_state_dict(torch.load(vae_path))


Epoch: 2 Train: 0.0306636, Test: 0.0278365, Lr:0.00025000
Epoch: 4 Train: 0.0185728, Test: 0.0153542, Lr:0.00050000
Epoch: 6 Train: 0.0078087, Test: 0.0064163, Lr:0.00075000
Epoch: 8 Train: 0.0045626, Test: 0.0042475, Lr:0.00100000
Epoch: 10 Train: 0.0031547, Test: 0.0037064, Lr:0.00099000
Epoch: 40 Train: 0.0008357, Test: 0.0012295, Lr:0.00073230
Epoch: 80 Train: 0.0003629, Test: 0.0005365, Lr:0.00048989
Epoch: 120 Train: 0.0002384, Test: 0.0005921, Lr:0.00032772
Epoch: 160 Train: 0.0001814, Test: 0.0002697, Lr:0.00021924
Epoch: 200 Train: 0.0001465, Test: 0.0002594, Lr:0.00014666
Epoch: 240 Train: 0.0001293, Test: 0.0002535, Lr:0.00009811
Epoch: 280 Train: 0.0001118, Test: 0.0002597, Lr:0.00006564
Epoch: 320 Train: 0.0001071, Test: 0.0002505, Lr:0.00004391
Early stopping!
✓ Saved VAE-MLP: final_trained_models/aTc_IPTG/2026-01-13_newmech_diff_params_VAEMLP_aTc_IPTG_TRF_Restrictive.pt

✓ Total training time for TRF_Restrictive: 10.0 minutes

TRAINING VAE-MLP FOR: Dogbox_Moderate

>>> T

/tmp/ipykernel_1883563/687396089.py:168: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vae_model_for_combo.load_state_dict(torch.load(vae_path))


Epoch: 2 Train: 0.0306911, Test: 0.0260252, Lr:0.00025000
Epoch: 4 Train: 0.0132053, Test: 0.0107140, Lr:0.00050000
Epoch: 6 Train: 0.0066831, Test: 0.0057347, Lr:0.00075000
Epoch: 8 Train: 0.0041727, Test: 0.0035818, Lr:0.00100000
Epoch: 10 Train: 0.0025430, Test: 0.0033245, Lr:0.00099000
Epoch: 40 Train: 0.0004772, Test: 0.0008345, Lr:0.00073230
Epoch: 80 Train: 0.0002671, Test: 0.0004392, Lr:0.00048989
Epoch: 120 Train: 0.0001770, Test: 0.0003402, Lr:0.00032772
Epoch: 160 Train: 0.0001491, Test: 0.0003924, Lr:0.00021924
Early stopping!
✓ Saved VAE-MLP: final_trained_models/aTc_IPTG/2026-01-13_newmech_diff_params_VAEMLP_aTc_IPTG_Dogbox_Moderate.pt

✓ Total training time for Dogbox_Moderate: 4.6 minutes

TRAINING VAE-MLP FOR: TRF_Moderate

>>> Training VAE...
VAE has 159,597 parameters
Epoch: 2 Train: 0.0051216, Test: 0.0020448, Lr:0.00025000
Epoch: 4 Train: 0.0012628, Test: 0.0009117, Lr:0.00050000
Epoch: 6 Train: 0.0009164, Test: 0.0005229, Lr:0.00075000
Epoch: 8 Train: 0.0007657, T

/tmp/ipykernel_1883563/687396089.py:168: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vae_model_for_combo.load_state_dict(torch.load(vae_path))


Epoch: 2 Train: 0.0322039, Test: 0.0280057, Lr:0.00025000
Epoch: 4 Train: 0.0171671, Test: 0.0139130, Lr:0.00050000
Epoch: 6 Train: 0.0073019, Test: 0.0065831, Lr:0.00075000
Epoch: 8 Train: 0.0046911, Test: 0.0033623, Lr:0.00100000
Epoch: 10 Train: 0.0029736, Test: 0.0028737, Lr:0.00099000
Epoch: 40 Train: 0.0006817, Test: 0.0008532, Lr:0.00073230
Epoch: 80 Train: 0.0003355, Test: 0.0005986, Lr:0.00048989
Early stopping!
✓ Saved VAE-MLP: final_trained_models/aTc_IPTG/2026-01-13_newmech_diff_params_VAEMLP_aTc_IPTG_TRF_Moderate.pt

✓ Total training time for TRF_Moderate: 3.9 minutes

✅ ALL VAE-MLPs TRAINED SUCCESSFULLY
   Trained 3 models


COMPARING PREDICTIONS ON EXPERIMENTAL DATA
This demonstrates parameter non-uniqueness doesn't affect predictions

>>> Testing TRF_Restrictive on experimental data...
  R² scores:
    aTc: R² = 0.9775
    IPTG: R² = 0.9804

>>> Testing Dogbox_Moderate on experimental data...
  R² scores:
    aTc: R² = 0.9697
    IPTG: R² = 0.9729

>>> Testing TRF_Moder